### 1 - Load and preprocess data

In [25]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# Load MovieLens ratings
ratings_df = pd.read_csv("../data/ml-latest-small/ratings.csv")
display(ratings_df.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [26]:
# Map userId and movieId to 0-based indices
user_id_to_index = {uid: idx for idx,
                    uid in enumerate(ratings_df["userId"].unique())}
movie_id_to_index = {mid: idx for idx,
                     mid in enumerate(ratings_df["movieId"].unique())}

ratings_df["userId"] = ratings_df["userId"].map(user_id_to_index)
ratings_df["movieId"] = ratings_df["movieId"].map(movie_id_to_index)

num_users = len(user_id_to_index)
num_movies = len(movie_id_to_index)

print(f"Users: {num_users}, Movies: {num_movies}")
ratings_df.head()

Users: 610, Movies: 9724


,userId,movieId,rating,timestamp
0,0,0,4.0,964982703
1,0,1,4.0,964981247
2,0,2,4.0,964982224
3,0,3,5.0,964983815
4,0,4,5.0,964982931


### 2 - Build Dataset and DataLoader

In [27]:
class MovieLensRatingDataset(Dataset):
    def __init__(self, ratings: pd.DataFrame):
        self.users = torch.tensor(ratings["userId"].values, dtype=torch.long)
        self.movies = torch.tensor(ratings["movieId"].values, dtype=torch.long)
        self.ratings = torch.tensor(
            ratings["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]


full_dataset = MovieLensRatingDataset(ratings_df)

In [28]:
# Train/test split (80/20)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

### 3 - Define Neural Collaborative Filtering model

In [29]:
import torch.nn as nn


class NeuralCollaborativeFiltering(nn.Module):
    def __init__(self, num_users: int, num_movies: int, embedding_dim: int = 50):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)

        self.fc_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user_ids, movie_ids):
        user_vec = self.user_embedding(user_ids)
        movie_vec = self.movie_embedding(movie_ids)
        x = torch.cat([user_vec, movie_vec], dim=1)
        return self.fc_layers(x).squeeze()

### 4 - Train the model

In [30]:
import torch.optim as optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NeuralCollaborativeFiltering(
    num_users, num_movies, embedding_dim=50).to(device)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 21
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch_users, batch_movies, batch_ratings in train_loader:
        batch_users, batch_movies, batch_ratings = (
            batch_users.to(device), batch_movies.to(
                device), batch_ratings.to(device)
        )

        optimizer.zero_grad()
        predictions = model(batch_users, batch_movies)
        loss = loss_fn(predictions, batch_ratings)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(batch_ratings)

    if epoch % 5 == 0:
        print(
            f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss/len(train_dataset):.4f}")

Epoch 1/21, Train Loss: 1.5106
Epoch 6/21, Train Loss: 0.6983
Epoch 11/21, Train Loss: 0.5938
Epoch 16/21, Train Loss: 0.4930
Epoch 21/21, Train Loss: 0.3901


### 5 - Evaluate on test set

In [31]:
from sklearn.metrics import mean_squared_error
import numpy as np

model.eval()
true_ratings, predicted_ratings = [], []
with torch.inference_mode():
    for batch_users, batch_movies, batch_ratings in test_loader:
        batch_users, batch_movies = batch_users.to(
            device), batch_movies.to(device)
        batch_predictions = model(batch_users, batch_movies).cpu().numpy()

        predicted_ratings.extend(batch_predictions)
        true_ratings.extend(batch_ratings.numpy())

rmse = np.sqrt(mean_squared_error(true_ratings, predicted_ratings))
print(f"Test RMSE: {rmse:.4f}")

Test RMSE: 0.9643


### 6 - Recommend Top-N movies for a user

In [32]:
movies_df = pd.read_csv("../data/ml-latest-small/movies.csv")
movieid_to_title = dict(zip(movies_df["movieId"], movies_df["title"]))


def recommend_top_movies_for_user(user_id: int, top_n: int = 5):
    model.eval()
    user_index = torch.tensor([user_id], dtype=torch.long).to(device)
    all_movies_tensor = torch.arange(num_movies, dtype=torch.long).to(device)

    user_tensor = user_index.repeat(num_movies)
    with torch.inference_mode():
        predictions = model(user_tensor, all_movies_tensor).cpu().numpy()

    top_indices = predictions.argsort()[::-1][:top_n]
    index_to_movieid = {v: k for k, v in movie_id_to_index.items()}
    # recommended_movie_ids = [index_to_movieid[i] for i in top_indices]
    
    recommendations = []
    for idx in top_indices:
        movie_id = int(index_to_movieid[idx])
        recommendations.append({
            "movie_id": movie_id,
            "title": movieid_to_title.get(movie_id, "Unknown Title"),
            "predicted_rating": float(predictions[idx])
        })
    
    return recommendations


print(recommend_top_movies_for_user(user_id=1, top_n=5))

[{'movid_id': 7312, 'title': 'Follow Me, Boys! (1966)', 'predicted_rating': 6.157899379730225}, {'movid_id': 69394, 'title': 'Stoning of Soraya M., The (2008)', 'predicted_rating': 6.149315357208252}, {'movid_id': 95761, 'title': 'Killer Joe (2011)', 'predicted_rating': 6.037369251251221}, {'movid_id': 146684, 'title': 'Cosmic Scrat-tastrophe (2015)', 'predicted_rating': 6.020081043243408}, {'movid_id': 52867, 'title': 'Ex, The (2007)', 'predicted_rating': 5.839355945587158}]


### 7 - Save artifacts

In [34]:
import pathlib

path_save = pathlib.Path("models")
path_save.mkdir(exist_ok=True)


torch.save(model.state_dict(), f"{path_save}/neural_cf_model.pth")
pd.Series(user_id_to_index).to_json(f"{path_save}/user_id_to_index.json")
pd.Series(movie_id_to_index).to_json(f"{path_save}/movie_id_to_index.json")